# 카메라 관찰과 시간별 기록 · v1.03

**2일차 7교시 · 수업 50분 + 휴식/전환 10분**

**목표:** 장면을 바꾸며 관찰하고 선택 프레임·촬영 시각·모델 결과를 연결합니다.

[전체 안내](README.md) · [Python 코드 읽기](../04.LLM_v2/PYTHON_GUIDE.md)

슬라이드로 개념을 먼저 보고 이 노트북을 실행합니다. **필수** 절을 위에서 아래로 실행하세요. 커널은 **Python (Day2 LangChain)**입니다. 다른 노트북의 변수는 자동으로 전달되지 않습니다.

셀 왼쪽 `[*]`는 실행 중이라는 뜻입니다. 응답을 기다린 뒤 다음 셀을 실행하세요. 요청 셀을 재실행하면 서버에 다시 요청합니다. 오류가 나면 그 셀에서 멈추고 오류 원문과 입력을 확인합니다.

**대응 슬라이드:** 47~52쪽 · [PDF 열기](../슬라이드/강의자료_2일차_v2.pdf#page=47)


## 준비 · 경로와 모델 객체

`Path`는 파일 경로를 다루는 도구입니다. 아래 셀은 현재 폴더부터 상위 폴더를 살펴 교재 위치를 찾습니다. `sys.path`에는 교재의 Python 모듈 위치를 추가합니다. 준비 코드는 제공하며 직접 외우지 않아도 됩니다.


In [ ]:
from pathlib import Path
import json
import os
import sys
from IPython.display import display, Image

# 현재 폴더에 교재가 없으면 부모 폴더를 차례로 확인합니다.
ROOT = Path.cwd().resolve()
while not (ROOT / "04.LLM_v2").is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError("NEPES 교재 폴더 안에서 노트북을 여세요.")
    ROOT = ROOT.parent
# import가 교재의 공통 모듈을 찾을 수 있도록 경로를 등록합니다.
sys.path.insert(0, str(ROOT / "04.LLM_v2"))
sys.path.insert(0, str(ROOT / "05.VLM_v2"))
print("교재 위치:", ROOT)
print("현재 Python:", sys.executable)


## 1. 카메라 촬영과 모델 분석의 속도는 다릅니다

카메라는 계속 프레임을 만들지만 모델은 한 요청을 처리하는 데 시간이 필요합니다. 모든 프레임을 대기열에 쌓으면 오래된 장면을 뒤늦게 분석하게 됩니다. 이번 프로그램은 최신 프레임 한 장을 선택합니다.

촬영은 별도 스레드, 분석 요청은 한 번에 하나입니다. 이 내부 코드는 제공합니다. 실습에서는 실제 선택된 장면과 기록을 관찰합니다.

![1. 카메라 촬영과 모델 분석의 속도는 다릅니다](assets/lesson-7-29.svg)


## camera_vlm.py 전체 코드 읽기

아래는 실행 파일의 전체 내용입니다. **코드 읽기용 블록**이며 argparse 진입점을 노트북에서 실행하지 않습니다. 실제 실행은 뒤의 Jupyter Terminal 명령을 사용합니다. 6교시의 main은 한 이미지를 처리했고, 이 스크립트의 main은 카메라 프레임을 반복 처리합니다.

| 구성 | 역할 | 읽을 때 확인할 점 |
|---|---|---|
| argparse | 입력 종류·모델·간격 설정 | `--model`은 별도 프로세스가 사용할 모델입니다. |
| CaptureWorker | 카메라를 별도 스레드에서 계속 읽기 | 모델 응답 대기 중에도 최신 프레임이 갱신됩니다. |
| LatestFrame | 최신 한 장과 번호를 보관 | 모델이 모든 프레임을 분석하는 것은 아닙니다. |
| observe / append_record | 모델 요청 / JSONL 저장 | 원본 경로·시각·응답·오류를 연결합니다. |
| finally | 카메라·스레드·창 정리 | 종료·오류·Ctrl+C에서도 자원을 정리합니다. |

`observe`는 공통 모듈의 함수이며 내부에서 image_message → model.invoke → 결과 검사를 실행합니다. `new_run`은 실행별 폴더, `append_record`는 JSONL 한 행을 만듭니다.

### 준비와 입력값 검사

양의 유한한 간격만 받습니다. import는 이름을 가져오며 실제 카메라 열기는 main에서 수행합니다.

```python
"""최신 USB/영상 프레임 한 장만 선택하여 LangChain VLM 관찰을 기록합니다.

7교시에서 제공하는 실행 프로그램입니다. 클래스·스레드를 직접 구현하는 과제는
아닙니다. 입력 열기 → 최신 프레임 선택 → 파일 저장 → VLM 요청 → JSONL 기록
순서로 읽으세요. 실제 실행은 Jupyter Terminal에서 합니다.
"""
import argparse
from datetime import datetime,timezone
import math
from pathlib import Path
import sys
import time
from threading import Event,Thread
sys.path.insert(0,str(Path(__file__).resolve().parents[1]))
from vlm_v2.core import make_model,new_run,observe,append_record,ROOT
from vlm_v2.sampling import LatestFrame


def positive(value):
    value=float(value)
    if not math.isfinite(value) or value<=0:raise argparse.ArgumentTypeError('양의 유한한 수가 필요합니다.')
    return value


# Thread를 상속한 클래스입니다. 카메라 읽기를 모델 응답 대기와 분리합니다.
# 모든 프레임을 저장하지 않고 LatestFrame에 최근 한 장을 덮어씁니다.
```

### CaptureWorker: 촬영을 계속하는 스레드

Event는 종료 신호입니다. frame.copy()로 프레임을 보관하고, 영상 입력은 FPS에 맞춰 읽습니다.

```python
class CaptureWorker(Thread):
    def __init__(self,capture,latest,stop,fps=None):
        super().__init__(daemon=True);self.capture,self.latest,self.stop,self.fps=capture,latest,stop,fps;self.error=None
    def run(self):
        try:
            while not self.stop.is_set():
                ok,frame=self.capture.read()
                if not ok:break
                # monotonic은 간격 계산용, UTC는 기록 간 비교용입니다.
                self.latest.put((frame.copy(),datetime.now(timezone.utc).isoformat()),time.monotonic())
                if self.fps:self.stop.wait(1/self.fps)
        except Exception as exc:self.error=exc
        finally:self.stop.set()
```

### main: 선택·추론·저장·종료

parse_args → 모델 준비 → 입력 열기 → 최신 프레임 선택 → 요청 간격 검사 → 저장·추론 → 종료 순입니다. save는 main 안에서 실행 설정을 공유하는 내부 함수입니다.

```python
def main():
    p=argparse.ArgumentParser(description=__doc__)
    # 모델 선택은 실행 인자로 받습니다. 7교시는 Gemma 4, 8교시는 Moondream입니다.
    p.add_argument('--model',default='gemma4:e2b-it-qat')
    p.add_argument('--text-output',action='store_true',help='JSON 스키마 없이 장면 설명을 기록합니다.')
    p.add_argument('--source',choices=['camera','video','images'],default='camera')
    p.add_argument('--camera-index',type=int,default=0)
    p.add_argument('--video',type=Path,default=ROOT/'Data/videos/05.VLM/ETS.mp4')
    p.add_argument('--images',nargs='+',type=Path)
    p.add_argument('--interval',type=positive,default=10)
    p.add_argument('--max-requests',type=int,default=3)
    p.add_argument('--preview',action='store_true')
    p.add_argument('--dry-run',action='store_true')
    a=p.parse_args()
    if a.max_requests<1:p.error('--max-requests는 1 이상입니다.')
    if a.preview and a.source=='images':p.error('이미지 목록에는 preview를 사용하지 않습니다.')
    # 별도 터미널 프로세스이므로 노트북의 model 변수를 재사용하지 않습니다.
    # 배포된 vision 모델을 요청할 객체를 여기에서 준비합니다.
    model=make_model(model=a.model,vision=True,num_predict=384)
    # Moondream 실습은 짧은 영어 질문과 자유 텍스트 응답으로 시작합니다.
    # JSON 생성 능력을 카메라·이미지 입력의 성공 조건으로 묶지 않습니다.
    structured=not (a.text_output or a.model.split(':')[0]=='moondream')
    question=None if structured else 'Describe only what is visible in this image in two short sentences.'
    # 매 실행마다 새 폴더를 만들어 이전 관찰을 덮어쓰지 않습니다.
    run=new_run('stream');log=run/'observations.jsonl'
    append_record(log,dict(type='header',source=a.source,interval=a.interval,model=model.model,dry_run=a.dry_run,structured=structured))
    print('기록:',log,flush=True)
    # **kwargs는 촬영 시각·프레임 번호 같은 추가 인자를 사전 형태로 받습니다.
    # observe가 돌려주는 raw는 모델 원문이며, 형식 오류가 나도 기록에 남깁니다.
    def save(path,**kwargs):
        record=observe(model,path,dry_run=a.dry_run,structured=structured,question=question,**kwargs);record['input_source']=a.source
        append_record(log,record)
        print(record['id'],record['status'],record['assessment'] or record['raw'] or record['error'] or 'dry-run',flush=True)
    if a.source=='images':
        if not a.images:p.error('--images 경로를 지정하세요.')
        import shutil
        for i,path in enumerate(a.images[:a.max_requests],1):
            target=run/f'input-{i:03d}{path.suffix}';shutil.copy2(path,target)
            save(target,sequence=i)
        return
    import cv2
    # source가 camera이면 장치 번호, video이면 파일 경로를 엽니다.
    cap=cv2.VideoCapture(a.camera_index if a.source=='camera' else str(a.video))
    if not cap.isOpened():cap.release();raise SystemExit('카메라/영상을 열 수 없습니다. 연결·장치 번호·중복 사용을 확인하세요.')
    fps=cap.get(cv2.CAP_PROP_FPS) if a.source=='video' else None
    if fps is not None and (not math.isfinite(fps) or fps<=0):fps=30
    latest,stop=LatestFrame(),Event();worker=CaptureWorker(cap,latest,stop,fps);worker.start()
    # seq는 프레임 번호, last_request는 직전 요청 시작 시각입니다.
    # -무한대로 시작하면 첫 프레임은 간격 조건을 바로 통과합니다.
    last_seq,last_request,count=0,-math.inf,0
    try:
        while count<a.max_requests:
            # 이미 본 프레임보다 새로운 한 장만 가져옵니다.
            item=latest.get_after(last_seq)
            if item is None:
                if stop.is_set():break
                time.sleep(.02);continue
            seq,captured_mono,(frame,captured_utc)=item;last_seq=seq
            if a.preview:
                cv2.imshow('VLM v2 latest frame',frame)
                if cv2.waitKey(1)&255==27:break
            # monotonic은 시스템 시각 보정의 영향을 받지 않는 간격 측정용 시계입니다.
            # interval은 최소 요청 시작 간격입니다. 추론이 느리면 실제 간격은 늘어납니다.
            if time.monotonic()-last_request<a.interval:
                time.sleep(.01);continue
            # 모델에 보낼 그 프레임을 먼저 저장합니다. 나중에 관찰 ID로 대조합니다.
            path=run/f'frame-{count+1:03d}.jpg'
            if not cv2.imwrite(str(path),frame):raise RuntimeError('프레임 저장 실패')
            last_request=time.monotonic()
            save(path,sequence=seq,captured_at=captured_utc);count+=1
        if worker.error:raise worker.error
    except KeyboardInterrupt:print('사용자 종료')
    finally:
        # 정상 종료·오류·Ctrl+C 모두 이 경로를 거쳐 카메라와 창을 정리합니다.
        stop.set();cap.release();worker.join(timeout=2)
        if a.preview:cv2.destroyAllWindows()
    print('완료:',log,flush=True)

if __name__=='__main__':main()
```

## 2. 실행 위치와 공동 실습 역할

카메라 프로그램은 **Jupyter Terminal**에서 실행합니다. 아래 명령을 셀에 Python으로 붙여 넣지 않습니다. 로컬 담당자 한 명만 카메라를 열고, 두 수강생이 장면 변경·관찰 기록 역할을 교대합니다.

노트북의 새 커널에서 model 객체를 만들었어도 터미널 프로세스에는 전달되지 않습니다. 카메라 스크립트가 별도로 배포 모델을 준비합니다. GUI는 Jetson에 연결한 모니터에서 봅니다.


### 사용 전 API 안내 · 카메라 CLI 인자

객체 만들기와 실제 실행을 구분해서 읽으세요. 아래 이름은 코드에 그대로 등장합니다.

| 함수·클래스 | 초기화·호출 예 | 입력 → 반환 / 주의점 |
|---|---|---|
| `--source` | `--source camera` 또는 `--source video` | 카메라 또는 제공 영상. 터미널에서 실행합니다. |
| `--dry-run` | `--max-requests 1 --dry-run` | 프레임·경로 확인. 모델 요청을 보내지 않습니다. |
| `--interval` / `--max-requests` | `--interval 10 --max-requests 3` | 최소 요청 시작 간격(초) / 최대 요청 수. 영상 종료 시 더 적을 수 있습니다. |
| `--preview` | `--preview` | 로컬 화면 미리보기. 원격 환경의 화면 표시 가능 여부를 확인합니다. |

먼저 **모델 요청 없는 dry-run**으로 카메라와 저장 경로를 확인합니다.

```bash
cd /workspace
Data/venvs/day2-langchain/bin/python 05.VLM_v2/scripts/camera_vlm.py --model gemma4:e2b-it-qat --source camera --max-requests 1 --dry-run
```

그다음 실제 관찰을 3회 수행합니다. 터미널에 출력되는 `observations.jsonl` 경로를 복사하세요.

```bash
Data/venvs/day2-langchain/bin/python 05.VLM_v2/scripts/camera_vlm.py --model gemma4:e2b-it-qat --source camera --interval 10 --max-requests 3 --preview
```

1. 첫 요청에는 원래 장면을 유지합니다.
2. 다음 요청 전에 사람 또는 물체의 보임·가림 상태를 바꿉니다.
3. 어떤 시각에 무엇을 바꾸었는지 사람이 별도로 기록합니다.
4. `Ctrl+C`로 중단할 수 있습니다. 중단·오류도 결과에 남깁니다.

카메라 연결이 어렵다면 **녹화 입력임을 표시**하고 실행합니다.

```bash
Data/venvs/day2-langchain/bin/python 05.VLM_v2/scripts/camera_vlm.py --model gemma4:e2b-it-qat --source video --interval 10 --max-requests 3
```

카메라 장애 해결에 5분 이상 걸리면 녹화 입력으로 전환합니다. 실제 카메라와 녹화 결과를 같은 종류의 관찰로 표시하지 않습니다.


## 3. 자신의 로그 파일을 명시적으로 선택합니다

JSONL은 한 줄에 JSON 사전 하나를 저장하는 형식입니다. 첫 줄에는 실행 설정이 있을 수 있고, 관찰 행에는 id·촬영 시각·원본 경로·raw·status가 있습니다.

다른 사람의 파일이나 가장 최근 파일을 자동 선택하지 않습니다. 자신이 방금 실행한 경로를 아래에 넣으세요. dry_run 행은 실제 VLM 관찰이 아닙니다.


### 사용 전 API 안내 · 관찰 로그 읽기

객체 만들기와 실제 실행을 구분해서 읽으세요. 아래 이름은 코드에 그대로 등장합니다.

| 함수·클래스 | 초기화·호출 예 | 입력 → 반환 / 주의점 |
|---|---|---|
| `load_observations` (교재 함수) | `records = load_observations(Path(OBSERVATION_LOG))` | JSONL 경로 → 관찰 사전 목록. LangChain API가 아닌 교재 유틸리티입니다. |
| 사전 조회 / 목록 슬라이스 | `row.get("status")`; `records[-3:]` | 없는 키는 기본값 None / 마지막 세 행. 오류 제거 전 시간창을 먼저 선택합니다. |

In [ ]:
from vlm_v2.core import load_observations

# 개인 설정: 터미널에 출력된 본인 로그의 실제 경로를 아래 기본값 ""에 넣으세요.
OBSERVATION_LOG = os.environ.get("OBSERVATION_LOG", "")
if not OBSERVATION_LOG:
    raise ValueError("실제 카메라 또는 녹화 입력의 observations.jsonl 경로를 입력하세요.")
log_path = Path(OBSERVATION_LOG)
records = load_observations(log_path)
print("선택한 파일:", log_path)
print("관찰 행 수:", len(records))


In [ ]:
for row in records:
    print("ID:", row["id"], "촬영 시각:", row["captured_at_utc"])
    print("상태:", row["status"], "입력 종류:", row.get("input_source"))
    print("모델 원문:", row.get("raw"))
    print("오류:", row.get("error"))


## 4. 답변에서 원본 프레임으로 돌아갑니다

모델의 문장만 읽지 말고 그 문장이 나온 원본을 확인합니다. 오류 행도 지우지 않습니다. 촬영 시각과 응답 완료 시각은 다르며, 응답이 늦게 왔다고 그때의 최신 장면을 설명한 것은 아닙니다.

![4. 답변에서 원본 프레임으로 돌아갑니다](assets/lesson-7-32.svg)


In [ ]:
# 첫 관찰의 원본을 확인합니다. 더 보고 싶으면 0을 1 또는 2로 바꿔보세요.
if not records:
    raise ValueError("관찰 행이 없습니다. 카메라/영상 실행 결과를 확인하세요.")
selected = records[0]
display(Image(filename=selected["input"]))
print(selected.get("raw"))
print(selected["captured_at_utc"])


## 5. 10초 간격과 실제 관찰 간격을 구분합니다

interval=10은 요청 시작 사이의 최소 간격입니다. 모델이 더 느리면 실제 간격은 더 길어집니다. 영상의 모든 프레임을 분석하는 설정이 아닙니다.

실행·관찰에 25분, 로그와 원본 확인에 10분을 사용합니다. 실제 기록의 요청 시각을 비교해 간격이 10초보다 길어지는 이유를 설명하세요.


## 완료 확인 · 직접 설명해보세요

- [ ] 실제 입력 종류와 로그 경로를 기록했다.
- [ ] 관찰 ID에서 원본 이미지까지 확인했다.
- [ ] 미관찰 구간과 모델 지연을 설명했다.

실행 성공과 의미 정답을 구분하세요. 답변이 이상하면 원문을 남기고, 어떤 입력 또는 단계에서 문제가 생겼는지 표시합니다.
